# Lab - Validating Extracted Data

The previous lab got Claude to return a clean dictionary. This one asks the harder
question: is what it says actually true?

| Task | What you learn |
| --- | --- |
| 1 | A value can fit the schema and still be wrong |
| 2 | When *not* to retry |
| 3 | Which cases a person should see |

**No API calls in this lab.** Every task is plain Python running against extractions
that already came back. That is not a simplification - it is the architecture. Claude
extracts, your application validates, and the validating half never talks to a model.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

It holds four extractions - the kind of dictionary the previous lab produced - along
with the customer message each one came from.

In [ ]:
# --- Lab setup (provided - just run it) ---
import json
import shopassist_lab
from shopassist_lab import check

# Our returns policy supports these four outcomes and nothing else.
ALLOWED_ACTIONS = ["refund", "replacement", "store_credit", "unclear"]

# Below this, our system is not confident enough to act on its own.
CONFIDENCE_THRESHOLD = 0.7

# Fields we cannot process a return without.
MUST_BE_PRESENT = ["order_id", "item"]


# Four extractions, exactly as the tool in the previous lab would return them.
# Every one of them is valid JSON with every required field. Read them.
CASES = [
    {
        "name": "clean",
        "customer_message": "My shoes ORD-12345678 arrived scratched. Please replace them. Photo attached.",
        "data": {
            "order_id": "ORD-12345678", "item": "shoes", "reason": "damaged_item",
            "desired_action": "replacement", "evidence_provided": True,
            "urgency": "normal", "confidence": 0.92,
            "conflict_detected": False, "conflict_reason": None,
            "missing_information": [],
        },
    },
    {
        "name": "wrong label",
        "customer_message": "The jacket ORD-11112222 is the wrong size. Can I swap it for a larger one?",
        "data": {
            "order_id": "ORD-11112222", "item": "jacket", "reason": "normal_return",
            "desired_action": "exchange", "evidence_provided": False,
            "urgency": "normal", "confidence": 0.88,
            "conflict_detected": False, "conflict_reason": None,
            "missing_information": [],
        },
    },
    {
        "name": "no order number",
        "customer_message": "My headphones stopped working after a week. I would like a refund.",
        "data": {
            "order_id": None, "item": "headphones", "reason": "damaged_item",
            "desired_action": "refund", "evidence_provided": False,
            "urgency": "normal", "confidence": 0.9,
            "conflict_detected": False, "conflict_reason": None,
            "missing_information": ["order_id"],
        },
    },
    {
        "name": "contradiction",
        "customer_message": "I want a refund for ORD-33334444, but if possible, just send me another one.",
        "data": {
            "order_id": "ORD-33334444", "item": "speaker", "reason": "damaged_item",
            "desired_action": "unclear", "evidence_provided": False,
            "urgency": "normal", "confidence": 0.45,
            "conflict_detected": True,
            "conflict_reason": "The customer mentions both refund and replacement.",
            "missing_information": [],
        },
    },
]

---

## Task 1 - Catch a value the schema should not have allowed

In [ ]:
# ============================================================
# TASK 1 - Catch a value the schema should not have allowed
# ============================================================
#
# WHAT TO DO
#   Write the check that reports a desired_action our returns
#   policy does not support.
#
# WHY IT MATTERS
#   Look at the "wrong label" case. It is valid JSON. Every
#   field is present. desired_action is a string. It is spelled
#   correctly. And "exchange" is not something our system can
#   do - there is no branch for it, no queue, no workflow.
#
#   A schema controls the shape of an answer. It cannot control
#   whether the answer means anything to your business.
#
#   This is the FIXABLE kind of error. The customer's message
#   does contain the answer - the model just filed it under a
#   name we do not use. That is what a retry with specific
#   feedback is for, and the cell prints one so you can see
#   what "specific" looks like.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Schema validation checks the shape of the output" - about
#   31 seconds in.
#
# HOW TO DO IT
#   One blank: the list of actions our policy supports. It is
#   defined in the setup cell as ALLOWED_ACTIONS and holds
#   "refund", "replacement", "store_credit" and "unclear".
# ============================================================

# BEGIN SOLUTION
def schema_errors(data):
    errors = []

    if data["desired_action"] not in ALLOWED_ACTIONS:
        errors.append(
            "desired_action is {!r}, which is not allowed".format(
                data["desired_action"]))

    return errors
# SCAFFOLD: def schema_errors(data):
# SCAFFOLD:     errors = []
# SCAFFOLD:
# SCAFFOLD:     if data["desired_action"] not in ...:   # ALLOWED_ACTIONS - the list from the setup cell
# SCAFFOLD:         errors.append(
# SCAFFOLD:             "desired_action is {!r}, which is not allowed".format(
# SCAFFOLD:                 data["desired_action"]))
# SCAFFOLD:
# SCAFFOLD:     return errors
# END SOLUTION: replace the ... below with the value named beside it


# Provided: turns a leftover blank into a sentence, instead of
# "argument of type 'ellipsis' is not a container".
try:
    schema_errors(CASES[0]["data"])
except TypeError:
    raise TypeError("You still have ... above. Replace it with ALLOWED_ACTIONS.")


for case in CASES:
    print("{:<18} {}".format(case["name"], schema_errors(case["data"]) or "no errors"))

# Provided: this is what a useful retry looks like. Four ingredients - the
# original message, the failed extraction, the exact error, and an instruction
# not to invent anything. "Please try again" would be none of them.
broken = next(c for c in CASES if schema_errors(c["data"]))

print()
print("--- retry prompt for the '{}' case ---".format(broken["name"]))
print("""
The field desired_action must be one of: {allowed}.
You returned {got!r}, which is not allowed.

Re-extract the return request from the original message.
Use "unclear" if the desired action is ambiguous.
Do not invent missing information.

Original message:
{message}

Your previous extraction:
{previous}
""".format(allowed=", ".join(ALLOWED_ACTIONS),
           got=broken["data"]["desired_action"],
           message=broken["customer_message"],
           previous=json.dumps(broken["data"], indent=2)))

check("schema_validation", schema_errors=schema_errors)

---

## Task 2 - Record what the message never said

In [ ]:
# ============================================================
# TASK 2 - Record what the message never said
# ============================================================
#
# WHAT TO DO
#   Write the check that lists the fields our system needs and
#   the extraction came back empty on.
#
# WHY IT MATTERS
#   Look at the "no order number" case. The customer genuinely
#   never wrote one. Nothing is broken here - the extraction is
#   correct.
#
#   So do not retry it. Re-prompting cannot produce an order
#   number that was never in the message; it can only produce an
#   invented one, and an invented order number is far worse than
#   an admitted gap. The right move is to record what is missing
#   and go ask the customer.
#
#   This is the line worth remembering from this lesson. Retry
#   when the model mishandled information that was there. When
#   it was never there, return null and say so.
#
# WHERE TO SEE IT IN THE LECTURE
#   "But retries do not help when the information is absent from
#   the source" - about 2 minutes 27 seconds in.
#
# HOW TO DO IT
#   Two blanks:
#
#       MUST_BE_PRESENT   the fields we cannot proceed without,
#                         from the setup cell
#       None              what a nullable field holds when the
#                         customer never gave a value
# ============================================================

# BEGIN SOLUTION
def missing_fields(data):
    gaps = []

    for field in MUST_BE_PRESENT:
        if data[field] is None:
            gaps.append(field)

    return gaps
# SCAFFOLD: def missing_fields(data):
# SCAFFOLD:     gaps = []
# SCAFFOLD:
# SCAFFOLD:     for field in ...:   # MUST_BE_PRESENT - the fields we cannot proceed without
# SCAFFOLD:         if data[field] is ...:   # None - what a nullable field holds when it is empty
# SCAFFOLD:             gaps.append(field)
# SCAFFOLD:
# SCAFFOLD:     return gaps
# END SOLUTION: replace each ... below with the value named beside it


# Provided: turns a leftover blank into a sentence.
try:
    missing_fields(CASES[0]["data"])
except TypeError:
    raise TypeError("You still have ... above. Replace them with MUST_BE_PRESENT "
                    "and None.")


for case in CASES:
    gaps = missing_fields(case["data"])
    print("{:<18} {}".format(case["name"], gaps or "nothing missing"))

check("missing_information", missing_fields=missing_fields)

---

## Task 3 - Route the risky cases to a person

In [ ]:
# ============================================================
# TASK 3 - Route the risky cases to a person
# ============================================================
#
# WHAT TO DO
#   Write the check that decides whether a case should be seen
#   by a human before anything happens to it.
#
# WHY IT MATTERS
#   Two things send a case to a person here, and neither is "the
#   model failed".
#
#   The first is a contradiction. Read the "contradiction" case:
#   the customer asks for a refund and then asks for a
#   replacement. There is no correct answer to automate, so the
#   extraction says so instead of picking one and hoping.
#
#   The second is low confidence - the system made a guess it is
#   not sure of.
#
#   Everything else goes through untouched, and that matters
#   just as much. A review queue that catches every case is a
#   review queue nobody reads.
#
# WHERE TO SEE IT IN THE LECTURE
#   "This brings us to human review routing" - about 5 minutes
#   24 seconds in.
#
# HOW TO DO IT
#   Two blanks:
#
#       data["conflict_detected"]   True when the message
#                                   contradicts itself
#       CONFIDENCE_THRESHOLD        0.7, from the setup cell -
#                                   below it we do not act alone
# ============================================================

# BEGIN SOLUTION
def needs_human_review(data):
    if data["conflict_detected"]:
        return True

    if data["confidence"] < CONFIDENCE_THRESHOLD:
        return True

    return False
# SCAFFOLD: def needs_human_review(data):
# SCAFFOLD:     if ...:   # data["conflict_detected"] - True when the message contradicts itself
# SCAFFOLD:         return True
# SCAFFOLD:
# SCAFFOLD:     if data["confidence"] < ...:   # CONFIDENCE_THRESHOLD - 0.7, from the setup cell
# SCAFFOLD:         return True
# SCAFFOLD:
# SCAFFOLD:     return False
# END SOLUTION: replace each ... below with the value named beside it


# Provided: turns a leftover blank into a sentence.
try:
    needs_human_review(CASES[0]["data"])
except TypeError:
    raise TypeError("You still have ... above. Replace them with "
                    "data[\"conflict_detected\"] and CONFIDENCE_THRESHOLD.")

# The summary below uses all three checks, so the earlier ones have to work too.
try:
    schema_errors(CASES[0]["data"])
    missing_fields(CASES[0]["data"])
except TypeError:
    raise TypeError("An earlier task still has ... in it. Fill in the previous cells "
                    "and run them again first.")


# Provided: the three checks together. The order is the decision - a value our
# system cannot use is worth fixing before anything else, a missing field means
# the customer has to be asked, and only then do we look at how risky the case is.
def decide(data):
    if schema_errors(data):
        return "retry with feedback"
    if missing_fields(data):
        return "ask the customer"
    if needs_human_review(data):
        return "human review"
    return "automate"


print("{:<18} {:<22} {}".format("case", "decision", "why"))
for case in CASES:
    data = case["data"]
    why = (schema_errors(data) or missing_fields(data)
           or (data["conflict_reason"] if data["conflict_detected"] else None)
           or ("confidence {}".format(data["confidence"])
               if data["confidence"] < CONFIDENCE_THRESHOLD else "-"))
    print("{:<18} {:<22} {}".format(case["name"], decide(data), why))

check("human_review", needs_human_review=needs_human_review)

---

## Done

Four extractions, four different outcomes, and not one of them was decided by a model.

```
clean              automate
wrong label        retry with feedback
no order number    ask the customer
contradiction      human review
```

Every one of those four dictionaries was valid JSON with every required field present.
The schema did its job perfectly and told you nothing about which of these you could
safely act on.

That is the shape to carry forward. **Claude extracts. Your application validates.**
Retry only when the problem is fixable, return null when information is missing, and
use conflict and confidence to decide when a person needs to look.

In production this layer grows - schema libraries, business rules, database checks,
policy checks, review queues. The architecture does not change.